# Preprocessing (HPC-optimised)

Extracts crops from a video based on trajectory files.
Key I/O optimisations:

- **Async write queue**: crops are written by background threads so the main loop never blocks on disk.
- **Zero PNG compression**: fastest possible writes (`cv2.IMWRITE_PNG_COMPRESSION, 0`).
- **Pre-built skip-set**: existing crop filenames are collected once with `glob` instead of one `exists()` syscall per frame.
- **Numpy crop buffer**: the white canvas is kept in CPU memory; only the current video frame is transferred GPU → CPU once per frame.


In [ ]:
from pathlib import Path
import threading
import queue
import cv2
import torch
import numpy as np
from tqdm import tqdm

## Device & Paths


In [ ]:
# Check whether a GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEBUG: Processing on device: {device}")

data_dir = Path("/scratch/cvcdt011/data")

## Configuration


In [ ]:
HALF_SIZE = 50  # Half-width/height of each crop in pixels
CROP_SIZE = 2 * HALF_SIZE

Z_THRESHOLD = 4.0  # Z-score threshold for outlier detection

NUM_WRITERS = 4  # Background writer threads
WRITE_QUEUE_MAX = 512  # Max crops buffered in memory before the main loop blocks

PNG_PARAMS = [cv2.IMWRITE_PNG_COMPRESSION, 3]  # Default compression level

# Train/Val/Test Split ratios
TRAIN_RATIO = 0.5
VAL_RATIO = 0.30
TEST_RATIO = 0.2
RANDOM_SEED = 42

## Helper: Read Next Row


In [ ]:
def read_next_row(fh):
    """Read and parse the next line from a trajectory file handle.
    Returns (frame_id, x, y, angle) or None if EOF.
    """
    line = fh.readline()
    if not line:
        return None
    parts = line.strip().split(",")
    return (int(parts[0]), float(parts[1]), float(parts[2]), float(parts[4]))

## Outlier Detection (Z-score for Position and Orientation)


In [ ]:
def detect_trajectory_outliers(file_path, z_threshold=Z_THRESHOLD):
    """Reads a trajectory file, computes displacement and angular change between consecutive steps,
    and returns a set of frame IDs flagged as outliers based on Z-scores.
    """
    rows = []
    with open(file_path, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) >= 5:
                rows.append(
                    (int(parts[0]), float(parts[1]), float(parts[2]), float(parts[4]))
                )

    if len(rows) < 3:
        return set()

    frames = [r[0] for r in rows]
    xs = np.array([r[1] for r in rows])
    ys = np.array([r[2] for r in rows])
    thetas = np.array([r[3] for r in rows])

    # 1. Position Jumps (Displacement)
    dx = np.diff(xs)
    dy = np.diff(ys)
    displacements = np.sqrt(dx**2 + dy**2)

    d_theta = np.diff(thetas)
    d_theta = (d_theta + 180) % 360 - 180
    abs_d_theta = np.abs(d_theta)

    outlier_frames = set()

    # Compute z-scores for displacements
    if len(displacements) > 1:
        mean_disp = np.mean(displacements)
        std_disp = np.std(displacements)
        if std_disp > 1e-6:
            z_disp = (displacements - mean_disp) / std_disp
            for idx in np.where(np.abs(z_disp) > z_threshold)[0]:
                outlier_frames.add(frames[idx + 1])

    # Compute z-scores for orientation changes
    if len(abs_d_theta) > 1:
        mean_theta = np.mean(abs_d_theta)
        std_theta = np.std(abs_d_theta)
        if std_theta > 1e-6:
            z_theta = (abs_d_theta - mean_theta) / std_theta
            for idx in np.where(np.abs(z_theta) > z_threshold)[0]:
                outlier_frames.add(frames[idx + 1])

    return outlier_frames

## Async Write Queue


In [ ]:
write_queue = queue.Queue(maxsize=WRITE_QUEUE_MAX)


def _writer_worker():
    """Background thread: pulls (path, ndarray) from the queue and writes to disk."""
    while True:
        item = write_queue.get()
        if item is None:  # Sentinel: shut down
            write_queue.task_done()
            break
        path, img = item
        try:
            cv2.imwrite(str(path), img, PNG_PARAMS)
        except Exception as e:
            print(f"DEBUG: Write error {path}: {e}")
        write_queue.task_done()


writer_threads = []
for _ in range(NUM_WRITERS):
    t = threading.Thread(target=_writer_worker, daemon=True)
    t.start()
    writer_threads.append(t)

print(f"DEBUG: Started {NUM_WRITERS} writer threads.")

## Preprocessing Setup


In [ ]:
video_path = data_dir / "rec1.mp4"
trajectory_dir = data_dir / "rec1_trajectories"
base_crops_dir = data_dir / "crops"
base_crops_dir.mkdir(parents=True, exist_ok=True)

txt_files = sorted(list(trajectory_dir.glob("*.txt")))
file_handles = []

# Shuffle and split trajectories to prevent data leakage (same trajectory does not span splits)
np.random.seed(RANDOM_SEED)
shuffled_files = list(txt_files)
np.random.shuffle(shuffled_files)

num_files = len(shuffled_files)
train_end = int(num_files * TRAIN_RATIO)
val_end = train_end + int(num_files * VAL_RATIO)

train_files = shuffled_files[:train_end]
val_files = shuffled_files[train_end:val_end]
test_files = shuffled_files[val_end:]

# Create a mapping from trajectory ID (stem) to its split
split_mapping = {}
for f in train_files:
    split_mapping[f.stem] = "train"
for f in val_files:
    split_mapping[f.stem] = "val"
for f in test_files:
    split_mapping[f.stem] = "test"

print(f"DEBUG: Found {len(txt_files)} trajectory files.")
print(
    f"DEBUG: Splits - Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}"
)

for f in txt_files:
    fh = open(f, "r")
    split = split_mapping[f.stem]
    file_handles.append((f.stem, fh))
    (base_crops_dir / split / f.stem).mkdir(parents=True, exist_ok=True)

## Precompute Outliers & Existing Crops


In [ ]:
# --- Outliers ---
outliers_dict = {}
total_outliers_flagged = 0
for f in txt_files:
    outliers = detect_trajectory_outliers(f)
    outliers_dict[f.stem] = outliers
    total_outliers_flagged += len(outliers)
print(
    f"DEBUG: Flagged {total_outliers_flagged} outlier frames across {len(txt_files)} trajectories."
)

# --- Existing crops: one glob per trajectory dir instead of per-frame exists() calls ---
existing_crops = {}
for f in txt_files:
    tid = f.stem
    split = split_mapping[tid]
    crop_dir = base_crops_dir / split / tid
    if crop_dir.exists():
        # Build a set of frame numbers that are already on disk for this bee
        # and also delete any that are outliers from previous runs
        existing = set()
        for p in crop_dir.iterdir():
            # filename format: frame_000123.png -> frame number 123
            try:
                frame_num = int(p.stem.split("_")[1])
            except IndexError, ValueError:
                continue
            if frame_num in outliers_dict[tid]:
                p.unlink()  # Remove leftover outlier crops from previous runs
            else:
                existing.add(frame_num)
        existing_crops[tid] = existing
    else:
        existing_crops[tid] = set()

print("DEBUG: Existing-crops index built.")

In [ ]:
# Pre-load first row of each trajectory
next_rows = {}
for tid, fh in file_handles:
    row = read_next_row(fh)
    if row:
        next_rows[tid] = row

## Main Crop Extraction Loop


In [ ]:
# Pre-allocate white canvas in CPU numpy memory (reused across all crops)
white_canvas = np.full((CROP_SIZE, CROP_SIZE, 3), 255, dtype=np.uint8)

cap = cv2.VideoCapture(str(video_path))
cap.set(cv2.CAP_PROP_BUFFERSIZE, 8)  # Increase video decoder prefetch buffer
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_count = 0
stats = {"skipped": 0, "processed": 0, "outliers_filtered": 0}
pbar = tqdm(total=total_frames, desc="Extracting crops")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w = frame.shape[:2]

        for traj_id, fh in file_handles:
            if traj_id not in next_rows or next_rows[traj_id][0] != frame_count:
                continue

            # Outlier: skip and do not save crop
            if frame_count in outliers_dict.get(traj_id, set()):
                stats["outliers_filtered"] += 1

            # Already on disk: skip
            elif frame_count in existing_crops.get(traj_id, set()):
                stats["skipped"] += 1

            # New crop: extract and enqueue for async write
            else:
                try:
                    x, y = (
                        int(round(next_rows[traj_id][1])),
                        int(round(next_rows[traj_id][2])),
                    )

                    # Desired window bounds
                    y_start = x - HALF_SIZE
                    y_end = x + HALF_SIZE
                    x_start = y - HALF_SIZE
                    x_end = y + HALF_SIZE

                    # Clamped bounds (intersection with frame)
                    ymin_f = max(0, y_start)
                    ymax_f = min(h, y_end)
                    xmin_f = max(0, x_start)
                    xmax_f = min(w, x_end)

                    # Copy white canvas and paste valid region
                    crop = white_canvas.copy()
                    crop[
                        ymin_f - y_start : ymax_f - y_start,
                        xmin_f - x_start : xmax_f - x_start,
                    ] = frame[ymin_f:ymax_f, xmin_f:xmax_f]

                    split = split_mapping[traj_id]
                    crop_filename = (
                        base_crops_dir
                        / split
                        / traj_id
                        / f"frame_{frame_count:06d}.png"
                    )
                    write_queue.put((crop_filename, crop))  # Non-blocking enqueue
                    stats["processed"] += 1
                except Exception as e:
                    print(
                        f"DEBUG: Error processing {traj_id} at frame {frame_count}: {e}"
                    )

            row = read_next_row(fh)
            if row:
                next_rows[traj_id] = row
            else:
                next_rows.pop(traj_id, None)

        pbar.update(1)
        if frame_count % 250 == 0:
            pbar.set_postfix(
                {
                    "processed": stats["processed"],
                    "skipped": stats["skipped"],
                    "outliers": stats["outliers_filtered"],
                }
            )
        frame_count += 1

finally:
    pbar.close()
    cap.release()
    for _, fh in file_handles:
        fh.close()

# Wait for all pending writes to finish, then shut down worker threads
write_queue.join()
for _ in writer_threads:
    write_queue.put(None)
for t in writer_threads:
    t.join()

print(
    f"DEBUG: Finished. Processed: {stats['processed']}, "
    f"Skipped: {stats['skipped']}, Outliers filtered: {stats['outliers_filtered']}"
)

## Outlier Visualization
Visualizes a trajectory and highlights the filtered outliers in red.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_outliers(traj_id):
    txt_file = trajectory_dir / f"{traj_id}.txt"
    rows = []
    with open(txt_file, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) >= 5:
                rows.append((int(parts[0]), float(parts[1]), float(parts[2]), float(parts[4])))
    
    if len(rows) < 3:
        return
        
    frames = np.array([r[0] for r in rows])
    xs = np.array([r[1] for r in rows])
    ys = np.array([r[2] for r in rows])
    thetas = np.array([r[3] for r in rows])
    
    dx = np.diff(xs)
    dy = np.diff(ys)
    displacements = np.sqrt(dx**2 + dy**2)
    
    d_theta = np.diff(thetas)
    d_theta = (d_theta + 180) % 360 - 180
    abs_d_theta = np.abs(d_theta)
    
    pos_outliers = set()
    ang_outliers = set()
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    
    # --- Position Analysis ---
    if len(displacements) > 1:
        mean_disp = np.mean(displacements)
        std_disp = np.std(displacements)
        if std_disp > 1e-6:
            z_disp = (displacements - mean_disp) / std_disp
            for idx in np.where(np.abs(z_disp) > Z_THRESHOLD)[0]:
                pos_outliers.add(frames[idx + 1])
            
            # Plot Threshold
            thresh_disp = mean_disp + Z_THRESHOLD * std_disp
            ax2.axhline(thresh_disp, color='red', linestyle='--', label=f'Threshold ({Z_THRESHOLD}σ)')
                
    # --- Angular Analysis ---
    if len(abs_d_theta) > 1:
        mean_theta = np.mean(abs_d_theta)
        std_theta = np.std(abs_d_theta)
        if std_theta > 1e-6:
            z_theta = (abs_d_theta - mean_theta) / std_theta
            for idx in np.where(np.abs(z_theta) > Z_THRESHOLD)[0]:
                ang_outliers.add(frames[idx + 1])
            
            # Plot Threshold
            thresh_theta = mean_theta + Z_THRESHOLD * std_theta
            ax3.axhline(thresh_theta, color='red', linestyle='--', label=f'Threshold ({Z_THRESHOLD}σ)')

    # --- Plotting Trajectory ---
    ax1.plot(xs, ys, label='Trajectory', color='blue', alpha=0.5, marker='.', zorder=1)
    
    # Scatter points for outliers
    pos_x = [xs[i] for i, f in enumerate(frames) if f in pos_outliers and f not in ang_outliers]
    pos_y = [ys[i] for i, f in enumerate(frames) if f in pos_outliers and f not in ang_outliers]
    if pos_x:
        ax1.scatter(pos_x, pos_y, color='red', label='Positional Outlier', zorder=5, s=50)
        
    ang_x = [xs[i] for i, f in enumerate(frames) if f in ang_outliers and f not in pos_outliers]
    ang_y = [ys[i] for i, f in enumerate(frames) if f in ang_outliers and f not in pos_outliers]
    if ang_x:
        ax1.scatter(ang_x, ang_y, color='orange', label='Angular Outlier', zorder=4, s=50)
        
    both_x = [xs[i] for i, f in enumerate(frames) if f in pos_outliers and f in ang_outliers]
    both_y = [ys[i] for i, f in enumerate(frames) if f in pos_outliers and f in ang_outliers]
    if both_x:
        ax1.scatter(both_x, both_y, color='purple', label='Both', zorder=6, s=70, marker='*')
    
    ax1.set_title(f'Trajectory {traj_id}')
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.legend()
    ax1.invert_yaxis()
    
    # --- Boxplots ---
    ax2.boxplot(displacements, vert=True)
    ax2.set_title('Displacements (px)')
    ax2.set_ylabel('Distance')
    ax2.set_xticks([1])
    ax2.set_xticklabels(['Delta Pos'])
    ax2.legend()
    
    ax3.boxplot(abs_d_theta, vert=True)
    ax3.set_title('Angular Changes (deg)')
    ax3.set_ylabel('Degrees')
    ax3.set_xticks([1])
    ax3.set_xticklabels(['Delta Theta'])
    ax3.legend()
    
    plt.tight_layout()
    plt.show()

# Execution
traj_with_outliers = [tid for tid, outs in outliers_dict.items() if len(outs) > 0]
if traj_with_outliers:
    print(f"Found {len(traj_with_outliers)} trajectories with outliers.")
    plot_outliers(traj_with_outliers[0])
else:
    print("No outliers found to visualize.")